# Ü8.1 — Inkrementell mit Job Bookmark (LÖSUNG, Notebook-Variante)

Notebook-Gegenstück zum Job-Skript `solution_orders_incremental.py`. Der
Bookmark-Schlüssel ist `transformation_ctx` auf Quelle, Mapping und Senke.

**Wichtig — Bookmarks sind ein Job-Runtime-Feature:** der Bookmark-Stand wird pro
**Job-Name** geführt und braucht `--job-bookmark-option job-bookmark-enable`. Im
Notebook setzen wir das über `%%configure` plus `Job.init(<name>)`/`job.commit()`;
primär gehört die Übung in den **Job** (das `.py`). Interaktiv dient das Notebook
dem Nachvollziehen der Code-Form.

**Voraussetzung:** `raw.orders` katalogisiert (Ü3.1).

Füll die `# TODO`-Stellen selbst; vergleiche danach mit `solution_orders_incremental.ipynb`.

In [ ]:
%idle_timeout 60
%glue_version 5.0
%worker_type G.1X
%number_of_workers 2
# In Glue Studio wird die IAM-Rolle in der UI gewählt. Alternativ per Magic:
# %iam_role arn:aws:iam::<account>:role/AWSGlueServiceRole-GfuGlueTraining
%%configure
{ "--enable-glue-datacatalog": "true", "--job-bookmark-option": "job-bookmark-enable" }

In [ ]:
from awsglue.context import GlueContext
from awsglue.job import Job
from awsglue.transforms import ApplyMapping
from pyspark.context import SparkContext

sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)

# Bookmark-Stand wird pro Job-Name gefuehrt:
job = Job(glueContext)
job.init("orders-incremental-nb")

OUTPUT = "s3://gfu-glue-training-<account>/processed/orders/"

## 1) Quelle — `transformation_ctx` trägt den Bookmark

In [ ]:
# TODO 1a: transformation_ctx="source_orders" ergaenzen (Bookmark-Schluessel).
source = glueContext.create_dynamic_frame.from_catalog(
    database="raw", table_name="orders"
)

## 2) ApplyMapping — den Bookmark-Kontext ergänzen

In [ ]:
# TODO 1b: transformation_ctx="mapped_orders" ergaenzen.
mapped = ApplyMapping.apply(
    frame=source,
    mappings=[
        ("cust id", "string", "customer_id", "string"),
        ("order total", "string", "order_total", "double"),
        ("order date", "string", "order_date", "string"),
        ("status", "string", "status", "string"),
    ],
)

## 3) Senke — den Bookmark-Kontext ergänzen

In [ ]:
# TODO 1c: transformation_ctx="sink_orders" ergaenzen.
sink = glueContext.getSink(
    path=OUTPUT,
    connection_type="s3",
    updateBehavior="UPDATE_IN_DATABASE",
    partitionKeys=["order_date"],
    enableUpdateCatalog=True,
)
sink.setCatalogInfo(catalogDatabase="processed", catalogTableName="orders")
sink.setFormat("glueparquet")
sink.writeFrame(mapped)

## 4) commit — schreibt den Bookmark-Stand fort

In [ ]:
job.commit()  # persistiert den Bookmark-Stand fuer den naechsten Lauf

## Test-Ablauf
1. Notebook einmal ausführen (verarbeitet `orders.csv`).
2. `seed/orders_2.csv` nach `raw/orders/` kopieren.
3. Erneut ausführen — nur `orders_2.csv` wird verarbeitet (Bookmark überspringt
   `orders.csv`).

Bookmark zurücksetzen: Glue-Konsole *Reset job bookmark* bzw.
`aws glue reset-job-bookmark --job-name orders-incremental-nb`.